# Animer — retrain taste vectors on 2023 data

Your current CF vectors come from a 2016 Kaggle dump, so **anime from 2019 onward have no taste signal at all** — recommendations for them run on synopsis similarity alone. This notebook trains new vectors on the **Anime Dataset 2023** (≈24M ratings from ≈270k MyAnimeList users) and tests them against the current ones *before* anything changes.

It compares two options:

| | what it does |
|---|---|
| **Retrain** | rebuild every vector from the 2023 data |
| **Fold-in** | keep every current vector exactly as it is, and only add vectors for anime that have none |

and a **popularity** baseline — a model that doesn't beat "just recommend what's popular" isn't learning taste.

**Nothing is written to Supabase unless you set `UPLOAD = True`**, the backup exists, and the new vectors actually test better. Your app code doesn't change either way: the same `cf_embedding` column, same `match_anime_cf` function.

**Before you start:** wake your Supabase project (free tier pauses when idle), and have your Kaggle account ready. Rough timings on free Colab: download 2–5 min · retrain 5–15 min · fold-in 3–5 min · test 1–2 min.

In [ ]:
!pip install -q kagglehub implicit supabase pandas numpy scipy tqdm

## 1. Settings

The defaults are sensible. `FACTORS` must stay 64 because the `cf_embedding` column is `vector(64)`.

In [ ]:
FACTORS          = 64     # must stay 64: cf_embedding is vector(64)
REGULARIZATION   = 0.05
ITERATIONS       = 20
ALPHA            = 4.0    # extra weight for a strong like (how far above the person's own average)
LIKE_MIN_SCORE   = 6      # a like must be >= this AND >= the person's own average score
MIN_USER_LIKES   = 5      # skip people with fewer likes than this (too little signal)
MIN_ANIME_LIKERS = 10     # skip anime with fewer fans than this (its vector would be noise)

EVAL_USERS       = 5000   # people used for the before/after test
HOLDOUT          = 0.2    # share of each test person's likes hidden from every model
FEW_SEEDS        = 5      # second test: recommend from someone's top 5 only (like manual mode)
RECENT_FROM      = 2018   # anime from this year on count as "recent" in the report
FOLD_ROUNDS      = 5      # fold-in refinement rounds (tested: 1 pass wrecks classics, 5 fixes it)
SEED             = 42

CHOICE           = None   # None = pick the better of 'retrain' / 'foldin' by test score
UPLOAD           = False  # nothing is written to Supabase until you flip this
FORCE            = False  # allow uploading even if the new vectors test worse (don't)

## 2. Credentials

In Colab, the cleanest way is the **key icon (Secrets)** in the left sidebar: add `SUPABASE_URL` and `SUPABASE_SERVICE_KEY`. Otherwise you'll be prompted. Kaggle will prompt for its own login when downloading.

In [ ]:
import os, getpass

def _secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name) or getpass.getpass(f'{name}: ')

os.environ['SUPABASE_URL'] = _secret('SUPABASE_URL')
os.environ['SUPABASE_SERVICE_KEY'] = _secret('SUPABASE_SERVICE_KEY')
print('Credentials loaded.')

## 3. Download the 2023 ratings

Only the three columns needed are loaded, so this fits comfortably in Colab's memory. Column names are detected rather than assumed.

In [ ]:
import json
import kagglehub
import numpy as np
import pandas as pd

if not (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')):
    kagglehub.login()

path = kagglehub.dataset_download('dbdmobile/myanimelist-dataset')
print('Files:', sorted(os.listdir(path)))
csv = os.path.join(path, 'users-score-2023.csv')

header = pd.read_csv(csv, nrows=0).columns
norm = {c: c.strip().lower().replace(' ', '_') for c in header}
def _col(*options):
    for c, n in norm.items():
        if n in options:
            return c
    raise KeyError(f'None of {options} found in columns {list(header)}')
U, A, R = _col('user_id'), _col('anime_id'), _col('rating', 'score')

df = pd.read_csv(csv, usecols=[U, A, R], dtype={U: 'int64', A: 'int64', R: 'float32'})
df.columns = ['user_id', 'anime_id', 'rating']
df = (df.dropna()
        .query('rating > 0')
        .drop_duplicates(['user_id', 'anime_id'], keep='last')
        .astype({'user_id': 'int32', 'anime_id': 'int32'})
        .reset_index(drop=True))

print(f'{len(df):,} ratings · {df.user_id.nunique():,} people · {df.anime_id.nunique():,} anime')
print(f'score range: {df.rating.min():g} to {df.rating.max():g}')

## 4. Define "liked" — per person

A harsh rater's 7 can be a rave; a generous rater's 7 a shrug. So a **like** is a score at or above *that person's own average*, and at least `LIKE_MIN_SCORE`. Everything else is not a like.

This is the main fix over the old model, which treated **every** rating as a like — a 2/10 counted as "liked, weakly". The printout shows how much of the data that affected.

In [ ]:
g = df.groupby('user_id').rating
df['u_mean'] = g.transform('mean').astype('float32')
u_std = g.transform('std').fillna(0).clip(lower=1.0).astype('float32')
df['like'] = (df.rating >= df.u_mean) & (df.rating >= LIKE_MIN_SCORE)
z = ((df.rating - df.u_mean) / u_std).clip(0, 3)
df['conf'] = (1.0 + ALPHA * z).astype('float32')
del u_std, z

print(f"Counted as likes now:             {df.like.mean():.0%} of ratings")
print(f"Below the person's own average:   {(df.rating < df.u_mean).mean():.0%}   <- the old model counted all of these as likes")
print(f"Scored 5 or lower:                {(df.rating <= 5).mean():.0%}")

## 5. Load your catalogue and the current vectors from Supabase

Needed to test the current vectors and to know which anime the app can actually recommend.

In [ ]:
from supabase import create_client

sb = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_SERVICE_KEY'])

rows, page, PAGE = [], 0, 1000
while True:
    res = (sb.table('anime')
             .select('id, mal_id, year, title, title_english, cf_embedding')
             .not_.is_('mal_id', 'null')
             .range(page * PAGE, (page + 1) * PAGE - 1)
             .execute())
    rows += res.data
    if len(res.data) < PAGE:
        break
    page += 1

def _vec(v):
    if v is None:
        return None
    if isinstance(v, str):
        v = json.loads(v)
    return np.asarray(v, dtype=np.float32)

cat = (pd.DataFrame([{'id': r['id'], 'mal_id': int(r['mal_id']), 'year': r['year'],
                      'title': r['title_english'] or r['title']} for r in rows])
         .drop_duplicates('mal_id').reset_index(drop=True))
OLD = {int(r['mal_id']): _vec(r['cf_embedding']) for r in rows if r['cf_embedding'] is not None}
print(f'Catalogue: {len(cat):,} anime · {len(OLD):,} currently have a CF vector')

## 6. Hide some favourites for the test

For `EVAL_USERS` people, a random `HOLDOUT` share of the anime they liked is **hidden from every model**. Later, each model recommends 10 anime and we count how many hidden favourites it found. Only anime in your catalogue are hidden, since those are the only ones the app can recommend.

In [ ]:
rng = np.random.default_rng(SEED)
in_cat = set(cat.mal_id)

def grouped(keys, *cols):
    # Fast group-by for large arrays: returns unique keys and each column split per key.
    order = np.argsort(keys, kind='stable')
    k = keys[order]
    bounds = np.flatnonzero(np.diff(k)) + 1
    uniq = k[np.r_[0, bounds]]
    return uniq, [np.split(c[order], bounds) for c in cols]

likes = df[df.like]
for _ in range(3):   # repeat: dropping people can push anime under the threshold, and vice versa
    per_user = likes.groupby('user_id').size()
    likes = likes[likes.user_id.isin(per_user[per_user >= MIN_USER_LIKES].index)]
    per_anime = likes.groupby('anime_id').size()
    likes = likes[likes.anime_id.isin(per_anime[per_anime >= MIN_ANIME_LIKERS].index)]
keep_users, keep_anime = set(likes.user_id.unique()), set(likes.anime_id.unique())

cat_likes = likes[likes.anime_id.isin(in_cat)]
per_user = cat_likes.groupby('user_id').size()
pool = per_user[per_user >= 10].index.to_numpy()
eval_users = rng.choice(pool, size=min(EVAL_USERS, len(pool)), replace=False)

heldout = {}
test = cat_likes[cat_likes.user_id.isin(set(eval_users.tolist()))]
users_, (items_,) = grouped(test.user_id.to_numpy(), test.anime_id.to_numpy())
for u, items in zip(users_, items_):
    k = max(1, int(round(len(items) * HOLDOUT)))
    heldout[int(u)] = set(rng.choice(items, size=k, replace=False).tolist())

KEY = 1_000_000   # anime ids are < 1e6, so user*KEY + anime is a unique pair key
hide = np.array([u * KEY + a for u, s in heldout.items() for a in s], dtype=np.int64)
pair = df.user_id.to_numpy(np.int64) * KEY + df.anime_id.to_numpy(np.int64)
train_df = df[df.user_id.isin(keep_users).to_numpy() & ~np.isin(pair, hide)]
train_likes = train_df[train_df.like & train_df.anime_id.isin(keep_anime)]
del pair

print(f'Training on {len(train_likes):,} likes · {train_likes.user_id.nunique():,} people · {train_likes.anime_id.nunique():,} anime')
print(f'Test: {len(heldout):,} people · {sum(map(len, heldout.values())):,} hidden favourites')

## 7. Option A — Retrain

A fresh ALS model on likes only, weighted by how far above their own average each person rated it.

In [ ]:
import implicit
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix

u_codes, u_index = pd.factorize(train_likes.user_id)
a_codes, a_index = pd.factorize(train_likes.anime_id)
M = csr_matrix((train_likes.conf.to_numpy(np.float32), (u_codes, a_codes)),
               shape=(len(u_index), len(a_index)))

model = AlternatingLeastSquares(factors=FACTORS, regularization=REGULARIZATION,
                                iterations=ITERATIONS, random_state=SEED,
                                use_gpu=implicit.gpu.HAS_CUDA)
model.fit(M)
if implicit.gpu.HAS_CUDA:
    model = model.to_cpu()

NEW = {int(m): np.asarray(model.item_factors[i], dtype=np.float32) for i, m in enumerate(a_index)}
print(f'Retrained: {len(NEW):,} anime have a vector · {sum(m in in_cat for m in NEW):,} of them are in your catalogue')

## 8. Option B — Fold-in

Keeps every current vector **exactly as it is** and only adds vectors for anime that have none.

1. **Place people on the existing map** from their likes of anime that already have a vector.
2. **Place the missing anime** where the people who loved them sit.
3. **Refine** for `FOLD_ROUNDS` rounds: people and the new anime adjust to each other while the current vectors stay frozen — the same back-and-forth real training does.

Step 3 matters. Tested on a simulated catalogue, a single pass made the new anime "generic" enough to crowd every classic out of the top 10 (classic hit rate 11.6% → 0.4%). Five rounds brought classics back to 9.6% while still adding recent anime.

It only works for people who liked both anime with and without a vector — they're the bridge between the two.

In [ ]:
def solve_side(fixed, idx_list, conf_list, reg):
    # Closed-form implicit-ALS step: vectors for one side with the other side fixed.
    # Checked against implicit's own output: direction match 0.9998.
    FtF = fixed.T @ fixed
    I = reg * np.eye(fixed.shape[1], dtype=np.float32)
    out = np.zeros((len(idx_list), fixed.shape[1]), dtype=np.float32)
    for n, (idx, c) in enumerate(zip(idx_list, conf_list)):
        F = fixed[idx]
        A = FtF + (F.T * (c - 1.0)) @ F + I
        out[n] = np.linalg.solve(A, F.T @ c)
    return out

old_ids = np.array([m for m in OLD if m in in_cat])
Y_old = np.stack([OLD[m] for m in old_ids]).astype(np.float32)
old_pos = {int(m): i for i, m in enumerate(old_ids)}
missing = in_cat - set(old_pos)

# Bridge people: liked >= 3 anime that have a vector, and >= 1 that doesn't.
n_old = train_likes[train_likes.anime_id.isin(set(old_pos))].groupby('user_id').size()
bridge = set(n_old[n_old >= 3].index) & set(train_likes[train_likes.anime_id.isin(missing)].user_id.unique())
b = train_likes[train_likes.user_id.isin(bridge)]

fans = b[b.anime_id.isin(missing)].groupby('anime_id').size()
new_ids = np.array(sorted(fans[fans >= MIN_ANIME_LIKERS].index))
new_pos = {int(m): len(old_ids) + i for i, m in enumerate(new_ids)}
all_pos = {**old_pos, **new_pos}
b = b[b.anime_id.isin(set(all_pos))]
bn = b[b.anime_id.isin(set(new_pos))]

def place_new(users, X):
    u_pos = {int(u): i for i, u in enumerate(users)}
    items, (idx_i, conf_i) = grouped(bn.anime_id.to_numpy(), bn.user_id.map(u_pos).to_numpy(),
                                     bn.conf.to_numpy(np.float32))
    assert np.array_equal(items, new_ids)
    return solve_side(X, idx_i, conf_i, REGULARIZATION)

# Step 1: people from the anime that already have a vector.
bo = b[b.anime_id.isin(set(old_pos))]
users, (idx_l, conf_l) = grouped(bo.user_id.to_numpy(), bo.anime_id.map(old_pos).to_numpy(),
                                 bo.conf.to_numpy(np.float32))
X = solve_side(Y_old, idx_l, conf_l, REGULARIZATION)
# Step 2: the missing anime from those people.
Y_new = place_new(users, X)
print(f'Placed {len(users):,} bridge people and {len(new_ids):,} new anime')

# Step 3: refine. Current vectors stay frozen throughout.
users_all, (idx_u, conf_u) = grouped(b.user_id.to_numpy(), b.anime_id.map(all_pos).to_numpy(),
                                     b.conf.to_numpy(np.float32))
for r in range(FOLD_ROUNDS):
    X = solve_side(np.vstack([Y_old, Y_new]), idx_u, conf_u, REGULARIZATION)
    Y_new = place_new(users_all, X)
    print(f'  refinement round {r + 1}/{FOLD_ROUNDS} done')

FOLD = dict(OLD)
FOLD.update({int(a): Y_new[i] for i, a in enumerate(new_ids)})
print(f'Fold-in: {len(FOLD):,} anime with a vector (current {len(OLD):,}, added {len(new_ids):,})')

## 9. The test

Some anime each test person loved were hidden. Each option tries to guess them from the rest of that person's list, the same way your app builds recommendations. **The percentage is how many hidden favourites it guessed.** Higher is better.

Two things keep it fair:

- **Sequels don't count.** Your app never recommends a sequel or prequel of something you've seen, so guessing *Attack on Titan Season 2* for a Season 1 fan earns no points. This uses the app's own franchise rule.
- **Tiny differences are treated as ties.** Every number has a `±`: how much it could wobble by chance. A difference smaller than that isn't real.

An option is picked only if **newer anime clearly improve**, **older anime stay at least 90% as good**, and **the overall score isn't clearly worse**.

In [ ]:
import re
from scipy.sparse import csr_matrix as _csr

cat_ids = cat.mal_id.to_numpy()
pos = {int(m): i for i, m in enumerate(cat_ids)}
years = pd.to_numeric(cat.year, errors='coerce').to_numpy()
is_recent = np.nan_to_num(years, nan=-1) >= RECENT_FROM
is_classic = np.nan_to_num(years, nan=10**6) < RECENT_FROM

# Franchise groups, using the app's own rule (src/app/api/recommend/v2/route.ts):
# same key after cutting subtitles and season markers, or the same first word of 4+ letters.
_STOP = {'the', 'a', 'an', 'my', 'no', 'to', 'is', 'of', 'and', 'in', 'for', 'wa', 'ga', 'ni'}
def _fkey(t):
    if not t:
        return ''
    t = re.split(r'[:：]| - | – ', t.lower())[0]
    t = re.sub(r'\b(season|part|cour)\s*\d+\b', ' ', t)
    t = re.sub(r'\b\d+(st|nd|rd|th)\s+season\b', ' ', t)
    t = re.sub(r'\b(2nd|3rd|final|movie|ova|specials?)\b', ' ', t)
    t = re.sub(r'\b(ii|iii|iv|v|vi|vii)\b', ' ', t)
    t = re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9 ]', ' ', t)).strip()
    return ' '.join([w for w in t.split(' ') if w and w not in _STOP][:2])

_titles = {}
for r in rows:
    _titles.setdefault(int(r['mal_id']), (r.get('title'), r.get('title_english')))
_vocab, _ti, _tj = {}, [], []
for i, m in enumerate(cat_ids):
    toks = set()
    for k in map(_fkey, _titles.get(int(m), (None, None))):
        if k:
            toks.add('k:' + k)
            w = k.split(' ')[0]
            if len(w) >= 4:
                toks.add('w:' + w)
    for t in toks:
        _ti.append(i)
        _tj.append(_vocab.setdefault(t, len(_vocab)))
TOK = _csr((np.ones(len(_ti), np.float32), (_ti, _tj)), shape=(len(cat_ids), max(1, len(_vocab))))

# What each test person has on their visible list, and the queries built from it
ev = train_df[train_df.user_id.isin(set(heldout)) & train_df.anime_id.isin(set(pos))]
ev_users = list(heldout)
ev_row = {u: i for i, u in enumerate(ev_users)}
r_ = ev.user_id.map(ev_row).to_numpy()
c_ = ev.anime_id.map(pos).to_numpy()
SHAPE = (len(ev_users), len(cat_ids))
W_full = _csr(((ev.rating - 6.5).to_numpy(np.float32), (r_, c_)), shape=SHAPE)
SEEN = _csr((np.ones(len(ev), np.float32), (r_, c_)), shape=SHAPE)
top = ev.sort_values(['user_id', 'rating'], ascending=[True, False]).groupby('user_id').head(FEW_SEEDS)
W_few = _csr(((top.rating - 6.5).to_numpy(np.float32),
              (top.user_id.map(ev_row).to_numpy(), top.anime_id.map(pos).to_numpy())), shape=SHAPE)

# Per person: which anime share a franchise with something they've seen (the app hides those),
# and which hidden favourites are still fair to guess.
BATCH = 500
BLOCKED, hidden_fair, n_sequels = {}, {}, 0
for s in range(0, len(ev_users), BATCH):
    n = min(BATCH, len(ev_users) - s)
    blk = ((SEEN[s:s + n] @ TOK) @ TOK.T).toarray() > 0
    BLOCKED[s] = blk
    for r in range(n):
        u = ev_users[s + r]
        keep = {a for a in heldout[u] if not blk[r, pos[a]]}
        n_sequels += len(heldout[u]) - len(keep)
        hidden_fair[u] = keep
print(f'{n_sequels:,} of {sum(map(len, heldout.values())):,} hidden favourites were sequels or prequels of '
      f"shows the person had seen. The app never recommends those, so they're left out.\n")

def model_matrix(vecs):
    V = np.zeros((len(cat_ids), FACTORS), np.float32)
    has = np.zeros(len(cat_ids), bool)
    for m, v in vecs.items():
        i = pos.get(int(m))
        if i is not None:
            V[i], has[i] = v, True
    return V, has

def _score(S, start):
    S[BLOCKED[start]] = -np.inf                                 # same franchise as a seen show
    S[SEEN[start:start + S.shape[0]].nonzero()] = -np.inf         # already seen
    top10 = np.argpartition(-S, 10, axis=1)[:, :10]
    out = {'all': [], 'classic': [], 'recent': []}
    for r in range(S.shape[0]):
        picks = top10[r][np.isfinite(S[r, top10[r]])]
        rec = set(cat_ids[picks].tolist())
        hid = hidden_fair[ev_users[start + r]]
        for era, mask in (('all', None), ('classic', is_classic), ('recent', is_recent)):
            h = hid if mask is None else {a for a in hid if mask[pos[a]]}
            if h:
                out[era].append(len(rec & h) / min(10, len(h)))
    return out

def evaluate(vecs, W):
    # One score per person (so options can be compared person by person), plus coverage.
    if vecs is None:
        pop = np.zeros(len(cat_ids), np.float32)
        counts = train_likes[train_likes.anime_id.isin(set(pos))].anime_id.map(pos).value_counts()
        pop[counts.index.to_numpy()] = counts.to_numpy()
        coverage = 1.0
    else:
        V, has = model_matrix(vecs)
        Vn = V / np.maximum(np.linalg.norm(V, axis=1, keepdims=True), 1e-9)
        Q = np.asarray(W @ V)
        coverage = float(has.mean())
    agg = {'all': [], 'classic': [], 'recent': []}
    for s in range(0, len(ev_users), BATCH):
        n = min(BATCH, len(ev_users) - s)
        if vecs is None:
            S = np.tile(pop, (n, 1))
        else:
            S = Q[s:s + n] @ Vn.T
            S[:, ~has] = -np.inf
            S[~np.any(Q[s:s + n] != 0, axis=1)] = -np.inf
        for k, v in _score(S, s).items():
            agg[k] += v
    return {k: np.asarray(v, dtype=np.float64) for k, v in agg.items()}, coverage

MODELS = {'popularity': None, 'current': OLD, 'foldin': FOLD, 'retrain': NEW}
LABEL = {'popularity': 'Popularity (no taste)', 'current': 'Current (what the app uses)',
         'foldin': 'Fold-in (keep + add)', 'retrain': 'Retrain (2023)'}
results, coverage = {}, {}
for name, vecs in MODELS.items():
    for mode, W in (('full list', W_full), ('top-5 favourites', W_few)):
        results[(name, mode)], coverage[name] = evaluate(vecs, W)

def mean_pm(x):
    return x.mean(), (1.96 * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0)
def diff_pm(m, era, mode='full list'):
    d = results[(m, mode)][era] - results[('current', mode)][era]    # same people, same order
    return mean_pm(d)

ERAS = (('classic', f'older (<{RECENT_FROM})'), ('recent', f'newer ({RECENT_FROM}+)'), ('all', 'overall'))
for mode in ('full list', 'top-5 favourites'):
    print('Recommending from their whole list' if mode == 'full list' else 'Recommending from their top 5 only')
    print(f"{'':30}" + ''.join(f'{lbl:>17}' for _, lbl in ERAS) + '   has a vector')
    for name in MODELS:
        cells = ''.join(f'{m:>10.1%} ±{h:.1%}' for m, h in (mean_pm(results[(name, mode)][e]) for e, _ in ERAS))
        print(f'{LABEL[name]:30}{cells}   {coverage[name]:>6.0%}')
    print()

def word(m, h):
    if m - h > 0: return 'real gain'
    if m + h < 0: return 'real loss'
    return 'a tie'
print('Compared with what the app uses now (whole list):')
for m in ('foldin', 'retrain'):
    parts = []
    for e, lbl in ERAS:
        d, h = diff_pm(m, e)
        parts.append(f'{lbl.split(" (")[0]} {d * 100:+.1f} ({word(d, h)})')
    print(f'  {LABEL[m]:28} ' + ' · '.join(parts))

cur_older = results[('current', 'full list')]['classic'].mean()
def check(m):
    reasons = []
    d, h = diff_pm(m, 'recent')
    if not d - h > 0:
        reasons.append("newer anime don't clearly improve")
    if results[(m, 'full list')]['classic'].mean() < 0.9 * cur_older:
        reasons.append('older anime lose more than 10%')
    d, h = diff_pm(m, 'all')
    if d + h < 0:
        reasons.append('overall is clearly worse')
    return reasons
REASONS = {m: check(m) for m in ('foldin', 'retrain')}
APPROVED = [m for m, r in REASONS.items() if not r]
CHOICE_FINAL = CHOICE or (max(APPROVED, key=lambda m: results[(m, 'full list')]['all'].mean()) if APPROVED else None)

print()
print(f'Verdict: use {LABEL[CHOICE_FINAL]}.' if CHOICE_FINAL else 'Verdict: keep what you have. Neither option passed.')
for m, r in REASONS.items():
    print(f"  {LABEL[m]}: {'passes' if not r else 'ruled out, ' + '; '.join(r)}")

# The upload cell from the first version of this notebook checks
# `best > cur and keeps_classics(CHOICE_FINAL)`. These make that check mean
# exactly "passed the test above", so an older upload cell still works.
keeps_classics = lambda m: m in APPROVED
cur = float(results[('current', 'full list')]['recent'].mean())
best = float(results[(CHOICE_FINAL, 'full list')]['recent'].mean()) if CHOICE_FINAL else cur

## 10. Sanity check — do the neighbours make sense?

Numbers can hide nonsense. Read these and ask: would a fan of the first show plausibly love these?

In [ ]:
assert CHOICE_FINAL, 'No option was chosen in the test above; nothing to check or upload.'
CHOSEN = {'retrain': NEW, 'foldin': FOLD}[CHOICE_FINAL]
V, has = model_matrix(CHOSEN)
Vn = V / np.maximum(np.linalg.norm(V, axis=1, keepdims=True), 1e-9)
title = dict(zip(cat.mal_id.astype(int), cat.title))

# Toradora!, Frieren, Your Name, Dandadan, Jujutsu Kaisen — mix of classic and recent
for m in [4224, 52991, 32281, 57334, 40748]:
    i = pos.get(m)
    if i is None or not has[i]:
        print(f'\n{title.get(m, m)}: no vector in the chosen model')
        continue
    sims = Vn @ Vn[i]
    sims[~has] = -np.inf
    sims[i] = -np.inf
    print(f'\n{title[m]}')
    for j in np.argsort(-sims)[:6]:
        print(f'   {sims[j]:.2f}  {title[int(cat_ids[j])]}')

_, has_old = model_matrix(OLD)
t = pd.DataFrame({'year': years, 'before': has_old, 'after': has})
t = t[(t.year >= 2014) & (t.year <= 2026)].groupby('year').agg(
    anime=('before', 'size'), before=('before', 'mean'), after=('after', 'mean'))
print('\nShare of catalogue with a taste vector, by year:')
print(t.to_string(formatters={'before': '{:.0%}'.format, 'after': '{:.0%}'.format}))

## 11. Back up the current vectors (one-time, in Supabase)

Open the **Supabase SQL Editor** and run this once. It's safe to re-run — it only copies if the backup column doesn't exist yet, so it can never overwrite the backup with new vectors.

```sql
do $$
begin
  if not exists (select 1 from information_schema.columns
                 where table_name = 'anime' and column_name = 'cf_embedding_2016') then
    alter table anime add column cf_embedding_2016 vector(64);
    update anime set cf_embedding_2016 = cf_embedding;
  end if;
end $$;
```

**To undo an upload later**, run:

```sql
update anime set cf_embedding = cf_embedding_2016;
```

In [ ]:
# What would be written. Nothing leaves this notebook yet.
assert CHOICE_FINAL, 'No option was chosen in the test above; nothing to upload.'
by_mal = {int(m): int(i) for m, i in zip(cat.mal_id, cat.id)}   # plain ints: numpy ints aren't JSON-serialisable
if CHOICE_FINAL == 'retrain':
    # Vectors from different trainings live in different spaces and can't be compared,
    # so every row gets the new vector, and rows the new model doesn't cover are cleared.
    writes = [{'id': by_mal[m], 'cf_embedding': (NEW[m].tolist() if m in NEW else None)}
              for m in by_mal if (m in NEW) or (m in OLD)]
else:
    # Fold-in keeps current vectors, so only the newly placed anime are written.
    writes = [{'id': by_mal[m], 'cf_embedding': FOLD[m].tolist()}
              for m in by_mal if m in FOLD and m not in OLD]

set_new = sum(1 for w in writes if w['cf_embedding'] is not None)
cleared = sum(1 for w in writes if w['cf_embedding'] is None)
print(f'{LABEL[CHOICE_FINAL]}: {len(writes):,} rows to write · {set_new:,} get a vector · {cleared:,} cleared')

## 12. Upload

Refuses unless: `UPLOAD = True`, the backup from step 11 exists and is complete, and the chosen model tested better than the current vectors.

In [ ]:
from tqdm import tqdm

assert UPLOAD, 'UPLOAD is False: nothing written. Review the test and sanity check, then set UPLOAD = True in Settings.'

try:
    chk = sb.table('anime').select('id', count='exact').not_.is_('cf_embedding_2016', 'null').execute()
except Exception as e:
    raise SystemExit(f'Backup column not found: run the SQL in step 11 first. ({e})')
assert chk.count == len(OLD), (
    f'Backup holds {chk.count:,} vectors but {len(OLD):,} are live now. '
    'If you already uploaded once, the "current" vectors ARE the uploaded ones; restore first if you want a clean comparison.')

assert CHOICE_FINAL in APPROVED or FORCE, (
    f'{LABEL[CHOICE_FINAL]} did not pass the test in step 9. Not uploading.')

BATCH, failed = 200, []
for i in tqdm(range(0, len(writes), BATCH), desc='Uploading'):
    batch = writes[i:i + BATCH]
    try:
        sb.table('anime').upsert(batch, on_conflict='id', returning='minimal').execute()
    except Exception as e:
        print(f'\nBatch {i} failed ({e}); retrying row by row')
        for r in batch:
            try:
                sb.table('anime').upsert([r], on_conflict='id', returning='minimal').execute()
            except Exception:
                failed.append(r['id'])

done = sb.table('anime').select('id', count='exact').not_.is_('cf_embedding', 'null').execute()
print(f'Done. Failed rows: {len(failed)} · anime with a CF vector now: {done.count:,}')

## 13. After uploading

1. In your project folder run `npm run eval` — your existing harness is the second opinion. Watch `modern_shounen` and `modern_mixed`, which used to fail.
2. Try the app with a few recent anime as seeds.
3. If anything looks worse, undo with the SQL in step 11. It takes seconds.